# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [10]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [11]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [12]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [15]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [16]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [17]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [18]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [19]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'external company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'capabilities page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [20]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'API endpoints', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [21]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [22]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 5 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
fishaudio/s2-pro
Updated
5 days ago
•
5.41k
•
512
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
9 days ago
•
67.6k
•
738
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
6 days ago
•
86.7k
•
322
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
13 days ago
•
238k
•
479
Lightricks/LTX-2.3
Updated
about 22 hours ago
•
597k
•
642
Browse 2M+ models
Spaces
Running
on
Zero
MC

In [ ]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [24]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [25]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nfishaudio/s2-pro\nUpdated\n5 days ago\n•\n5.41k\n•\n512\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n9 days ago\n•\n67.6k\n•\n738\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n6 days ago\n•\n86.7k\n•\n322\nHauhauCS/Qwen

In [26]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [27]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Hugging Face Brochure

## Who We Are
**Hugging Face** is the AI community building the future. We are the collaboration platform for machine learning (ML) enthusiasts, engineers, scientists, and developers worldwide. Our mission is simple yet powerful: to empower the next generation of machine learning experts and end users to create, share, and innovate in an open, ethical, and collaborative environment.

## What We Offer
- **Hugging Face Hub:** A central place for hosting and discovering over 2 million machine learning models, along with 500,000+ datasets and 1 million+ AI applications.  
- **Models:** Access a wide variety of state-of-the-art ML models, including popular ones updated frequently by our community.  
- **Datasets:** Explore extensive datasets to train, test, and validate machine learning applications.  
- **Spaces:** Deploy and share ML-powered applications and AI demos in an interactive manner.  
- **Buckets:** AI-native object storage designed for scalable and efficient storage of ML assets.  
- **Open-Source Stack:** Utilize our open-source tools to accelerate your ML workflow and collaboration.

## Our Community
At Hugging Face, collaboration is at the core. We are a fast-growing, passionate community where anyone can:
- Share models, datasets, and applications
- Discover and experiment with cutting-edge AI technology
- Learn and grow together through open knowledge exchange

Our platform encourages collaboration to build an open and ethical AI future. The community actively maintains and updates popular models, datasets, and apps, ensuring innovation remains accessible and fresh.

## Our Customers and Users
Hugging Face serves a diverse range of users including:
- Machine learning engineers and developers building AI applications  
- Researchers and scientists advancing AI technology  
- Enterprises seeking AI solutions and collaboration tools  
- Educators and students involved in AI and data science learning  

Our platforms cater to everyone from hobbyists and enthusiasts to industry leaders looking for reliable open-source AI resources and community support.

## Careers and Culture
- **Innovative and Inclusive:** Join a community-driven company at the forefront of AI technology with an inclusive, open culture fostering creativity and ethical practices.  
- **Growth and Learning:** Collaborate with experts worldwide and contribute to projects impacting the future of AI.  
- **Opportunities:** We encourage individuals passionate about machine learning, open-source development, and AI ethics to explore career opportunities with us. (Visit Hugging Face careers page for current openings.)  

## Why Choose Hugging Face?
- Trusted platform with millions of models, datasets, and applications  
- Largest collaborative machine learning community globally  
- Rapid innovation powered by open-source and community-driven tools  
- Commitment to ethical AI and responsible technology development  

---

Join Hugging Face today and be part of the AI community building the future.  
Explore, create, and collaborate at: **https://huggingface.co**

---

## Brand Assets
- Colors: #FFD21E (yellow), #FF9D00 (orange), #6B7280 (gray)  
- Logos Available in SVG, PNG, AI formats  

---

**Hugging Face** — The home of open, ethical, and collaborative machine learning.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community and collaboration platform dedicated to building the future of machine learning. It serves as a vibrant hub where machine learning engineers, researchers, and enthusiasts from around the world create, discover, share, and collaborate on models, datasets, and applications. Hugging Face is committed to fostering an open, ethical, and inclusive AI ecosystem that accelerates innovation and learning.

---

## Our Platform

- **Models**: Access and contribute to a repository of over 2 million machine learning models ranging from simple audio processors to state-of-the-art language models. These models are continuously updated and refined by the community.

- **Datasets**: Explore over 500,000 datasets supporting a wide variety of AI tasks. Users can upload and share datasets, enabling faster training and evaluation workflows.

- **Spaces**: Build, share, and demo AI applications easily using Spaces, which host community-developed AI-powered apps—like video generators and image editors—accessible to developers and users alike.

- **Storage Buckets**: Utilize AI-native object storage designed to seamlessly integrate with machine learning workloads for efficient data handling.

- **Open-Source Stack**: The platform is powered by a robust open-source infrastructure that enables rapid development, experimentation, and deployment of machine learning models.

---

## Our Community & Culture

The heart of Hugging Face is its fast-growing, collaborative, and global AI community. Hugging Face empowers the next generation of machine learning engineers, scientists, and end users to collectively develop innovative solutions and share knowledge. The culture is built on:

- **Openness**: Sharing models, datasets, and ideas freely with the world.
- **Collaboration**: Encouraging partnerships across disciplines and geographies.
- **Ethics**: Committing to responsible AI development and transparency.
- **Innovation**: Supporting cutting-edge research and practical applications to push AI boundaries.

---

## Customers & Use Cases

Hugging Face serves a diverse audience ranging from individual AI researchers and hobbyists to large enterprises. Key users include:

- AI researchers accelerating their experiments using ready-made models and datasets.
- Developers building AI-powered apps for various industries such as healthcare, finance, media, and more.
- Enterprises leveraging Hugging Face’s technology and ecosystem to deploy scalable, customizable AI solutions.
- Educators and students seeking an accessible environment for learning and experimentation.

---

## Careers at Hugging Face

Join a visionary company that is shaping the AI landscape! Hugging Face is looking for talented individuals passionate about open-source AI, machine learning, and community-building. Career opportunities include roles in:

- Machine Learning Research and Engineering
- Software Development and Infrastructure
- Data Science and AI Ethics
- Developer Relations and Community Management
- Product Management and Design

At Hugging Face, you will be part of a dynamic team committed to innovation, collaboration, and making AI accessible to everyone.

---

## Contact & Learn More

Explore the future of AI with Hugging Face today.  
Visit: [huggingface.co](https://huggingface.co)  

- Browse 2 million+ models  
- Access over 500k datasets  
- Discover industry-leading AI applications  

Together, let's build an open, ethical AI future.

---

**Hugging Face** — The AI community building the future.  
*Colors: #FFD21E, #FF9D00, #6B7280*

---

*This brochure summarizes Hugging Face's platform, community, and opportunities to engage in the growing AI revolution.*

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

## About Hugging Face
Hugging Face is a pioneering AI community dedicated to building the future of machine learning (ML). Their platform serves as a vibrant collaboration hub where machine learning engineers, data scientists, researchers, and AI enthusiasts come together to create, share, and innovate on models, datasets, and applications.

Recognized as *the home of machine learning*, Hugging Face empowers the next generation of ML professionals and end users to connect, learn, and advance open and ethical AI technologies.

---

## What Hugging Face Offers

### The Collaboration Platform
- **Model Hub:** Access over **2 million** models ranging from natural language processing to computer vision and audio processing.
- **Datasets:** Browse and contribute to over **500,000** datasets facilitating machine learning experimentation and development.
- **Spaces:** Deploy and discover **1 million+** AI-powered applications and demos that bring models to life with interactive interfaces.
- **Storage Buckets:** AI-native object storage designed to enhance ML infrastructure usability and efficiency.

### Open Source Focus
Hugging Face champions open source innovation through the **HF Open Source Stack**, enabling faster development cycles and community-driven progress.

---

## Community & Customers
Hugging Face hosts a thriving global community with projects ranging from academic research to enterprise AI solutions. Their ecosystem is built on democratizing AI access, welcoming hobbyists, startups, established companies, and academic institutions alike.

The platform features trending models like Qwen-series large language models, audio processing tools, and multimedia generators accessible to practitioners of all skill levels.

---

## Company Culture
Rooted deeply in openness, collaboration, and ethical AI development, Hugging Face fosters an inclusive and innovative culture where contributors and users can freely share knowledge and build together.

Key cultural elements include:
- Strong community engagement and transparency
- Commitment to building ethical and responsible AI
- Support for learning and professional growth within the AI and ML field

---

## Careers at Hugging Face
Hugging Face is continuously expanding its team of talented engineers, researchers, and product experts who are passionate about AI and open source software.

Working at Hugging Face means engaging in cutting-edge ML projects, contributing to a global community, and shaping the future of AI technologies in an inclusive and supportive environment.

---

## Join Hugging Face Today
- Explore and deploy models and datasets on the go
- Collaborate with one of the fastest-growing AI communities
- Build and showcase AI applications via Spaces
- Help create an open, ethical AI future

**Website:** [huggingface.co](https://huggingface.co)  
**Sign Up:** Create an account to join the platform and start collaborating today!

---

Hugging Face – *The AI community building the future.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>